# TRAIN candidate headroom — 2.000 ảnh, T4 ×2, không huấn luyện
Kiểm tra exact A có sinh được caption tốt hơn greedy hay không trước khi xây preference bank.
Một notebook dùng hai GPU inference độc lập. 500 ảnh đầu đo tốc độ; nếu dự toán vượt ngân sách thì dừng,
nếu đạt chạy đủ2.000ảnh TRAIN. Mỗiảnh1greedy+4sample(temperature.8/top-p.95),max30token gồm BOS.
Không train, không optimizer, không đánh giá val/test và không tự mở rộng10k.

**T4×2 · Internet ON · Save & Run All.** Thêm5dataset: pair-source-fullframe-ce,prompt-cache,
visual-cache,mscoco-2014,objectdetectionecache. Không cần H/L hoặc UPT hidden cache.
Gửi `preference_candidate_train2000_review.zip`; giữ toàn bộ folder để dùng lại nếu phương pháp được chốt.
Oracle headroom TRAIN không phải kết quả cải thiện mô hình. Novelty vẫn chưa được xác nhận.

In [ ]:
print('TRAIN CANDIDATE AUDIT START — NO TRAINING',flush=True)
from pathlib import Path
CHECKPOINT=Path('/kaggle/input/datasets/ducanh2403/pair-source-fullframe-ce/upt_hidden_caption_A/checkpoint.bin')
LEGACY_PROMPT=Path('/kaggle/input/datasets/ducanh2403/prompt-cache/prompt_clip_tokens_cache.pt')
VISUAL_CACHE=Path('/kaggle/input/datasets/ducanh2403/visual-cache')
COCO_JSON=Path('/kaggle/input/datasets/vuthetam/mscoco-2014/dataset_coco.json')
DETECTIONS=Path('/kaggle/input/datasets/ducanh2403/objectdetectionecache/objectdetectioncache.json')
OUTPUT=Path('/kaggle/working/preference_candidate_train2000')
STAGE=Path('/tmp/preference_candidate_train2000.h5')
for p in (CHECKPOINT,LEGACY_PROMPT,COCO_JSON,DETECTIONS):assert p.is_file(),p
assert VISUAL_CACHE.is_dir(),VISUAL_CACHE
assert not OUTPUT.exists() and not STAGE.exists(),'Preserve old results; use a fresh session/path.'


In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, tempfile, zlib
os.environ.update(PYTHONUNBUFFERED='1', HDF5_USE_FILE_LOCKING='FALSE', PYTHONHASHSEED='42',
    CUBLAS_WORKSPACE_CONFIG=':4096:8', OMP_NUM_THREADS='2', OPENBLAS_NUM_THREADS='2', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_XET='1')
# Isolate each bootstrap: embedded tracked overlays from a prior run are intentional edits.
REPO = Path(tempfile.mkdtemp(prefix='Image_Captioning-hidden-preference-candidate-2000-'))/'repo'
LOG = Path('/tmp/upt_hidden_preference-candidate-2000.log')
PROJECT_COMMIT = '34ab92ce3c926666ec71a9162d7169e69849f123'
def run(args, cwd=None):
    args = list(map(str, args))
    print('\nRunning:', ' '.join(args), flush=True)
    with LOG.open('a', encoding='utf-8') as log:
        log.write('\nRunning: ' + ' '.join(args) + '\n'); log.flush()
        with subprocess.Popen(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
    if status:
        raise RuntimeError(f'Command failed ({status}). Saved log: {LOG}')
def checkout(url, path, revision):
    if not path.exists():
        run(['git','clone',url,path])
    assert (path/'.git').is_dir(), path
    assert not subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=path,text=True).strip(), 'Preserve tracked edits; use a fresh runtime path.'
    run(['git','fetch','--depth','1','origin',revision], cwd=path)
    run(['git','checkout','--detach',revision], cwd=path)
checkout('https://github.com/Supzxjee/Image_Captioning.git', REPO, PROJECT_COMMIT)
SOURCES = json.loads(zlib.decompress(base64.b64decode('...')).decode('utf-8'))
SOURCE_HASHES = {'candidate_data.py': '772c9b979bf34b6f0199297ea7876b13a3b95c46701be4516c1ba2ef76dc9342', 'run_candidates.py': '37eaabac259750493d88fd545f263a41f35de19534cdd77d95cf004c4155048e', 'test_candidates.py': '0435f6db2187dfa7548845e9b07e06fc0fbecb41a434e452dc50d314578068b3', 'evidence_data.py': '7043d11c63d7ebfec8c8ed9e81eb951b9954399ff5939d146c35bc6e7ecfdc4e', 'lexical_policy.json': 'fd13046512449449375f94ecda44237a33d5a9223813d036fc6d76f4294abfab', 'captioning/models.py': '18026877836052c83e75c0d896adcc89822bb35616173c7e8b14b17c9a22e9d8', 'captioning/tokenizer.py': 'df2ace0025aa7d4e12c289ab7bca68046b48d4b6c2b5c12606a19ff5a50a638e', 'captioning/config.py': '4766d1bf82bf82b5199616c7d90ca74242f011e26f08db21b154b1c7d58e72bc', 'captioning/semantic_prompts.py': 'a06f12dea63604f9db9d19b6922815db27a35b648a17059407ffaf26a2a0d511', 'captioning/visual_cache.py': 'adbcc9678fff499fc767cdbf04a4966186a40729f51c5bfb2525e79f9b6b5fe1', 'captioning/scst_runtime.py': '3452ccfbd43876dc6381b78d1229d1d6b05fe723e3863a04f31c4a784497f669', 'captioning/scst.py': '3a9a1f32fb27dae0447066e1388a31461f3f0781d52db5861880b037a572a78e', 'captioning/object_consistency.py': '5fbd75b0ff4a35f0475d460a294894009d6007f468e248a0e4b528aaa8fabf73'}
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == SOURCE_HASHES[name], name
    destination = REPO/name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding='utf-8')
print('Pinned runtime and hash-checked embedded source READY', flush=True)


In [ ]:
run([sys.executable,'-m','pip','install','-q','transformers==4.57.1','h5py','numpy','Pillow','pycocoevalcap==1.2','nltk==3.9.2'])
run(['java','-version'])
run([sys.executable,'-u','test_candidates.py'],cwd=REPO)
import torch
assert torch.cuda.device_count()==2 and all('T4' in torch.cuda.get_device_name(i) for i in range(2)),'Select T4x2.'
CONFIG=Path('/tmp/preference_candidate_config.json')
cfg=dict(checkpoint=str(CHECKPOINT),legacy_prompt=str(LEGACY_PROMPT),visual_cache=str(VISUAL_CACHE),
    dataset_json=str(COCO_JSON),detections=str(DETECTIONS),output=str(OUTPUT),stage=str(STAGE),
    labels_archive=str(REPO/'experiments/noun_evidence_proposal/data/noun_evidence_train10k_v1_dataset.zip'),
    project_commit=PROJECT_COMMIT,implementation_sha256=SOURCE_HASHES,reported_remaining_gpu_hours=50,
    worker_budget_seconds=3*3600,scope='TRAIN2000 candidate feasibility only; no training/no val/test',
    generation=dict(greedy=1,samples=4,temperature=.8,top_p=.95,max_length_including_BOS=30))
CONFIG.write_text(json.dumps(cfg,indent=2),encoding='utf-8');PROTOCOL=OUTPUT/'protocol.json'
try:
    run([sys.executable,'-u','run_candidates.py','prepare','--config',CONFIG],cwd=REPO)
    run([sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2',
        '--max_restarts=0','run_candidates.py','worker','--config',PROTOCOL],cwd=REPO)
    run([sys.executable,'-u','run_candidates.py','evaluate','--config',PROTOCOL],cwd=REPO)
finally:
    if OUTPUT.is_dir():
        if LOG.is_file():shutil.copyfile(LOG,OUTPUT/'notebook.log')
        if not PROTOCOL.is_file():shutil.copyfile(CONFIG,OUTPUT/'requested_config.json')
        for name,source in SOURCES.items():
            path=OUTPUT/'source'/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(source,encoding='utf-8')
        import zipfile
        with zipfile.ZipFile(str(OUTPUT)+'_review.zip','w',zipfile.ZIP_DEFLATED) as z:
            for p in sorted(OUTPUT.rglob('*')):
                if p.is_file():z.write(p,p.relative_to(OUTPUT))
result=json.loads((OUTPUT/'numeric_summary.json').read_text(encoding='utf-8'))
assert result['status']=='CANDIDATE_AUDIT_COMPLETE','ZIP creation alone is not success.'
print(json.dumps(result,ensure_ascii=False,indent=2),flush=True)
print('SEND:',str(OUTPUT)+'_review.zip',flush=True)
print('KEEP:',OUTPUT,flush=True)
